# Persistence and reproducible continuation

Compare file sizes and round-trip RNG continuation. HDF5 stores a JSON envelope, not a native transition matrix.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
import time
model = markovify.load_model(OUT / "model.json.gz", random_state=SEED)
rows = []
for name, kwargs in [("benchmark.json",{}),("benchmark.json.gz",{}),("benchmark.h5",{}),("benchmark_compressed.h5",{"compression":"gzip"})]:
    path = OUT / name
    start = time.perf_counter()
    model.save(path, metadata={"seed":SEED}, **kwargs)
    save_time = time.perf_counter()-start
    start = time.perf_counter()
    restored = markovify.load_model(path)
    load_time = time.perf_counter()-start
    assert restored.chain.model == model.chain.model
    assert restored.chain.rng.getstate() == model.chain.rng.getstate()
    rows.append({"format":name,"bytes":path.stat().st_size,"save_seconds":save_time,"load_seconds":load_time})
stats = pd.DataFrame(rows)
display(stats)
stats.plot.bar(x="format",y="bytes",legend=False,title="Serialized model sizes")
plt.ylabel("Bytes")
plt.show()

In [ ]:
model.save(OUT / "checkpoint.json.gz")
expected = [model.make_sentence(test_output=False) for _ in range(20)]
restored = markovify.load_model(OUT / "checkpoint.json.gz")
assert expected == [restored.make_sentence(test_output=False) for _ in range(20)]
print("RNG continuation verified")